# 07 — Inference & Model Export

**Project:** Ecommerce-LLM-Finetuning
**Goal of this notebook:**
1. Load the base model + LoRA adapter
2. Run an interactive chatbot loop
3. Merge LoRA weights into the base model
4. Save the merged, standalone model for deployment (Streamlit app / HF Hub)

Run this after `05_finetune_llm.ipynb` (and optionally `06_evaluation.ipynb`).


In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import Config
from src.inference import EcommerceSupportBot
from src.utils import get_logger

logger = get_logger("notebook07")
cfg = Config()


## 1. Load base model + LoRA adapter

In [ ]:
bot = EcommerceSupportBot(cfg)
model, tokenizer = bot.load_adapter_model()
print("Loaded base model + adapter for inference.")


## 2. Generate answers for a few sample questions

In [ ]:
sample_questions = [
    "Where is my order #48213?",
    "How do I return a pair of shoes that don't fit?",
    "My coupon code SAVE20 isn't applying at checkout.",
    "Can you help me book a flight to Paris?",  # off-topic, should be politely declined
]

for q in sample_questions:
    answer = bot.generate(q, temperature=0.6, max_new_tokens=150, use_chat_template=False)
    print("Q:", q)
    print("A:", answer)
    print("-" * 80)


## 3. Interactive chatbot loop

Run the cell below and type questions. Type `exit` or `quit` to stop.
(In Colab, this uses `input()` — works in the standard notebook UI.)

In [ ]:
def interactive_chat(bot, max_turns=20):
    print("Ecommerce Support Assistant — type 'exit' to stop.\n")
    for _ in range(max_turns):
        user_msg = input("You: ")
        if user_msg.strip().lower() in {"exit", "quit"}:
            print("Assistant: Thanks for chatting — have a great day!")
            break
        reply = bot.generate(user_msg, temperature=0.7, max_new_tokens=200, use_chat_template=False)
        print(f"Assistant: {reply}\n")

# Uncomment to run interactively:
# interactive_chat(bot)


## 4. Merge LoRA adapter into the base model

Merging produces a standalone model that can be loaded with plain `transformers` (no PEFT/Unsloth dependency needed at serve time) — this is what the Streamlit app loads by default.

In [ ]:
merged_dir = bot.merge_and_save(cfg.merged_model_dir)
print(f"Merged model saved to: {merged_dir}")
!ls -lh {merged_dir}


## 5. Verify the merged model loads and generates correctly

In [ ]:
from src.inference import EcommerceSupportBot as _Bot

verify_bot = _Bot(cfg)
verify_bot.load_merged_model(cfg.merged_model_dir)

test_answer = verify_bot.generate(
    "I never received a confirmation email after placing my order, what should I do?",
    temperature=0.6,
)
print(test_answer)


## 6. (Optional) Push merged model to the Hugging Face Hub

Useful for deploying the Streamlit app on Streamlit Community Cloud / Spaces without re-uploading large weights manually.

In [ ]:
PUSH_TO_HUB = False  # flip to True and set your repo id + token to enable
HF_REPO_ID = "your-username/ecommerce-support-llm"

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # will prompt for an HF token (free account)

    verify_bot.model.push_to_hub(HF_REPO_ID)
    verify_bot.tokenizer.push_to_hub(HF_REPO_ID)
    print(f"Pushed merged model to https://huggingface.co/{HF_REPO_ID}")
else:
    print("Skipping Hub push (PUSH_TO_HUB=False).")


## Summary

- Loaded base model + LoRA adapter and ran sample + interactive inference
- Confirmed the off-topic guardrail behavior (system prompt) works as expected
- Merged the adapter into a standalone model saved at `models/merged_model/`
- Verified the merged model loads and generates correctly with plain `transformers`
- Optionally pushed the merged model to the Hugging Face Hub

**Next:** run the Streamlit app locally:
```bash
streamlit run app/streamlit_app.py
```